# 普莱费尔密码 (Playfair Cipher) — 自学课程

**分类**：古典密码学

**内容简介**：基于5×5密钥方阵的双字母替换密码，按三类规则加密，一战英军曾广泛使用



## 学习目标

- 理解算法规则与数学表达
- 实现加解密函数，并写基本测试
- 通过小实验观察安全性与攻击方法
- 能解释常见踩坑并独立调试



## 前置知识与符号约定

- 字母表：仅使用大写 A-Z
- 映射：$A\rightarrow 0,\dots,Z\rightarrow 25$
- 模运算：$x\bmod 26$ 结果落在 $\{0,\dots,25\}$

> 如果你希望支持中文/标点，本课程也会在后续练习中给出扩展思路。



In [ ]:
# 课程通用工具：字母映射与规范化
import string
from collections import Counter, defaultdict
import math, random

ALPHABET = string.ascii_uppercase
A2I = {ch:i for i,ch in enumerate(ALPHABET)}
I2A = {i:ch for i,ch in enumerate(ALPHABET)}

def normalize(text: str) -> str:
    """仅保留 A-Z 并转大写"""
    return ''.join(ch for ch in text.upper() if ch in ALPHABET)

def keep_nonletters_encrypt(text: str, enc_func):
    """对字母加密，非字母原样保留（用于扩展练习）"""
    out=[]
    for ch in text:
        if ch.upper() in ALPHABET:
            out.append(enc_func(ch.upper()))
        else:
            out.append(ch)
    return ''.join(out)

print(normalize("Hello, World! 123"))
# 预期输出：HELLOWORLD



# Step 1：把算法写成数学模型

我们用统一抽象描述密码：

- 加密：$E_k: \mathcal{P}\to\mathcal{C}$
- 解密：$D_k: \mathcal{C}\to\mathcal{P}$
- 正确性：

$$D_k(E_k(p))=p$$

这能帮助你：
- 写出可测试的实现
- 分析密钥空间大小
- 讨论攻击模型（已知明文、选择明文等）



## 自检小测

1) 什么是“模 26”运算？为什么它会让结果回到 0..25？
2) 为什么实现中必须统一 A→0 的映射？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



> 常见错误 / 踩坑点 / 调试提示：
>
> - 不要混用 A→1 与 A→0 两种映射。
> - 记得对 k 做 k%=26；否则 k>26 时结果可能不符合预期。
> - 先写 round-trip 测试：decrypt(encrypt(p,k),k)==p。



# Step 2：普莱费尔密码概念与规则

普莱费尔基于 5×5 方阵进行二元组替换。常用 I/J 合并。



## 自检小测

1) 为什么要把 I/J 合并？这会带来什么歧义？
2) 为什么要把明文分成二元组而不是单字母？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



# Step 2：普莱费尔密码原理

普莱费尔（Playfair）是基于 5×5 字母表的双字母（digraph）替换密码。

常见约定：把 I/J 合并（因此 25 个字母）。规则要点：

1) 先用关键字生成 5×5 方阵
2) 明文分成二元组；若同字母成对则插入填充字母（常用 X）
3) 加密规则：同一行、同一列、矩形替换

> 踩坑点：I/J 合并、填充规则、末尾补齐规则必须一致，否则解密还原会混乱。



In [ ]:
# Step 2：构建 5x5 方阵（I/J 合并）
import string

ALPH25 = "ABCDEFGHIKLMNOPQRSTUVWXYZ"  # 无 J

def playfair_square(key: str):
    key = normalize(key).replace("J","I")
    seen=set()
    seq=[]
    for ch in key + ALPH25:
        if ch == "J":
            ch = "I"
        if ch in ALPH25 and ch not in seen:
            seen.add(ch)
            seq.append(ch)
    square=[seq[i*5:(i+1)*5] for i in range(5)]
    pos={square[r][c]:(r,c) for r in range(5) for c in range(5)}
    return square, pos

sq, pos = playfair_square("MONARCHY")
for row in sq:
    print(row)
# 预期输出：一个 5x5 的字母方阵



In [ ]:
# Step 3：二元组分割（带填充）
def playfair_digraphs(text: str, filler="X"):
    t = normalize(text).replace("J","I")
    i=0
    pairs=[]
    while i < len(t):
        a=t[i]
        b=t[i+1] if i+1<len(t) else None
        if b is None:
            pairs.append((a, filler))
            i += 1
        elif a == b:
            pairs.append((a, filler))
            i += 1
        else:
            pairs.append((a, b))
            i += 2
    return pairs

print(playfair_digraphs("BALLOON"))
# 预期输出：例如 [('B','A'),('L','X'),('L','O'),('O','N')]（可能略有差异）



In [ ]:
# Step 4：加解密实现
def playfair_encrypt_pair(a,b,sq,pos):
    ra,ca = pos[a]
    rb,cb = pos[b]
    if ra == rb:
        return sq[ra][(ca+1)%5], sq[rb][(cb+1)%5]
    if ca == cb:
        return sq[(ra+1)%5][ca], sq[(rb+1)%5][cb]
    return sq[ra][cb], sq[rb][ca]

def playfair_decrypt_pair(a,b,sq,pos):
    ra,ca = pos[a]
    rb,cb = pos[b]
    if ra == rb:
        return sq[ra][(ca-1)%5], sq[rb][(cb-1)%5]
    if ca == cb:
        return sq[(ra-1)%5][ca], sq[(rb-1)%5][cb]
    return sq[ra][cb], sq[rb][ca]

def playfair_encrypt(text: str, key: str) -> str:
    sq,pos = playfair_square(key)
    out=[]
    for a,b in playfair_digraphs(text):
        x,y = playfair_encrypt_pair(a,b,sq,pos)
        out.extend([x,y])
    return ''.join(out)

def playfair_decrypt(text: str, key: str) -> str:
    sq,pos = playfair_square(key)
    t = normalize(text).replace("J","I")
    if len(t)%2==1:
        raise ValueError("密文长度必须为偶数")
    out=[]
    for i in range(0,len(t),2):
        a,b = t[i], t[i+1]
        x,y = playfair_decrypt_pair(a,b,sq,pos)
        out.extend([x,y])
    return ''.join(out)

ct = playfair_encrypt("INSTRUMENTS", "MONARCHY")
print(ct)
print(playfair_decrypt(ct, "MONARCHY"))
# 预期输出：解密应还原到带填充的形式（可能包含 X）



# Step 5：关于“去填充”的讨论

普莱费尔解密后常含填充字母（如 X）。是否删除 X 依赖语境：

- 如果明文中本来可能出现 X，盲删会误删
- 更稳妥的方法是：记录插入位置；或在更高层协议里携带长度/校验

> 调试提示：把“格式化规则”写成独立函数，便于复用与测试。



## 练习

1) 让代码支持自定义合并字母规则（例如合并 I/J 或 U/V）。
2) 写一个 `pretty_square(square)` 以更美观地打印方阵。
3) 思考：为什么普莱费尔比单字母替换更能抵抗频率分析？用“双字母频率”解释。



# Step 6：攻击直觉与局限

普莱费尔比单字母替换更抗频率分析，但仍可能被：
- 已知明文/选择明文
- 统计二元组频率与启发式搜索

> 现代密码使用多轮迭代与非线性结构来抵抗此类统计攻击。



## 总结与延伸

- 你已经完成：规则→数学→实现→测试→攻击/评估。
- 下一步可以：
  - 为实现添加更多字符集与格式化规则
  - 写更强的评分器做自动破译
  - 把多个古典密码组合，体验“组合不等于安全”

> 重要：古典密码主要用于学习思想；真实系统请使用经过标准化与审计的现代密码算法。



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。

